# Prompt Evaluation

This chapter I'll explore prompt evaluation, understand how it differs from prompt engineering.

## Prompt Engineering vs Prompt Evaluation

While Prompt Engineering is a set of practices when writing to achieve the effective prompts, Prompt Evaluation is about measuring how effective the prompt is in an automated way.

There are three ways to handle writing a prompts:
1. Write a prompt, test it once and decide it's good enough;
2. Write a prompt, test it a few times and adjust it to handle edge cases;
3. Run the prompt through an **evaluation pipeline**. This is what I'll look into.

There's something called "Evaluation-First Approach", that I can relate to something like test driven development. It takes time to setup all the testing infrastructure, but the payoff is reliability on the final applicaiton.

## Evaluation Workflow

We can split a workflow in five main steps:
1. Draft a prompt;
2. Create and eval dataset;
3. Feed through the model;
4. Feed through a grader;
5. Change prompt and repeat.

After running a workflow like that, we are able to grade the prompt.

The score will be the average of all the scores we got when running the prompt against the **evaluation dataset**.

In [8]:
from constants import MODEL
from litellm import completion, Message, Response

messages: list[Message] = []

def add_message(message: str, role: str):

    messages.append(Message(content=message, role=role))
def add_user_message(message: str):
    messages.append(Message(content=message, role="user"))

def add_assistant_message(message: str):
    messages.append(Message(content=message, role="assistant"))

def send_prompt(stop) -> Response:
    response = completion(
        model=MODEL,
        messages=messages,
        stop=stop
    )
    messages.append(response.choices[0].message)
    return response

def print_response(response: Response):
    print(response.choices[0].message.content)

def clear_context():
    global messages
    messages = []


### Generating a test dataset

We can use the model to create an eval dataset for us.

In [34]:
import json

def gen_dataset() -> dict:
    prompt = """
Generate a test dataset for prompt evaluation. The dataset will evaluate prompts that assist with math related questions.

Example output:
[
    {
        "question": "Some questions about math"
    },
    ...more questions
]

The answer should be plain JSON, with no aditional commentary, just the raw JSON string.

* Focus on simple questions about linear algebra
* Focus on questions that don't require long winded explanations
* Focus on questions that don't require calculations
* Foocus on questions about theory

Generate 3 objects.
"""
    message = Message(
        role="user",
        content=prompt
    )
    response = completion(
        model=MODEL,
        messages=[message]
    )
    raw_json = response.choices[0].message.content
    return json.loads(raw_json)

dataset = gen_dataset()
dataset

[{'question': 'What is the definition of a linear transformation in linear algebra?'},
 {'question': 'What is the significance of eigenvalues in the context of matrix operations?'},
 {'question': 'How does the concept of orthogonality apply to vectors in linear algebra?'}]

In [35]:
with open('datasets/prompt_evaluation.json', 'w') as f:
    json.dump(dataset, f, indent=2)

### Running the Evaluation

This is the step where we'll create the main loop that goes through each object from the dataset and runs it along with the prompt we drafted.

In [36]:

def run_eval(prompt, dataset):
    results = []

    score = 10 # this will be hardcoded for now

    for test_case in dataset:
        new_prompt = prompt + test_case["question"]
        response = completion(model=MODEL, messages=[Message(role="user", content=new_prompt)])
        result = {
            "output": response.choices[0].message.content,
            "test_case": new_prompt,
            "score": score
        }
        results.append(result)

    return results

In [37]:
with open("datasets/prompt_evaluation.json", "r") as f:
    dataset = json.load(f)

prompt_draft = "Answer this math related question:\n"

results = run_eval(prompt_draft, dataset)

In [40]:
with open("results/prompt_evaluation.json", "w") as f:
    json.dump(results, f, indent=2)

Ok, that's pretty much it, basically a flow to iterate over each test case, merge it with my prompt, and run it against the model.

I'll clean up a bit by refactoring into more function to handle each portion separately.

In [41]:
def run_prompt(test_case):
    prompt = f"""Answer this math related question:\n{test_case['question']}"""

    response = completion(model=MODEL, messages=[Message(role="user", content=prompt)])
    return response.choices[0].message.content

def eval_test_case(test_case):
    score = 10 # hardcoded

    output = run_prompt(test_case)
    result = {
        "test_case": test_case,
        "output": output,
        "score": score,
    }

    return result

def eval_dataset(dataset):
    return [eval_test_case(test_case) for test_case in dataset]

In [42]:
with open("datasets/prompt_evaluation.json", "r") as f:
    dataset = json.load(f)

results = eval_dataset(dataset)

In [43]:
results

[{'test_case': {'question': 'What is the definition of a linear transformation in linear algebra?'},
  'output': 'A **linear transformation** (also called a **linear map** or **linear operator**) is a function $ T: V \\rightarrow W $ between two vector spaces $ V $ and $ W $ over the same field (e.g., real or complex numbers) that satisfies the following two properties for all vectors $ \\mathbf{u}, \\mathbf{v} \\in V $ and all scalars $ c $:\n\n1. **Additivity**:  \n   $$\n   T(\\mathbf{u} + \\mathbf{v}) = T(\\mathbf{u}) + T(\\mathbf{v})\n   $$\n\n2. **Homogeneity**:  \n   $$\n   T(c\\mathbf{u}) = cT(\\mathbf{u})\n   $$\n\nThese properties ensure that $ T $ preserves the structure of the vector space, including vector addition and scalar multiplication. Linear transformations are fundamental in linear algebra and can be represented by matrices when $ V $ and $ W $ are finite-dimensional.',
  'score': 10},
 {'test_case': {'question': 'What is the significance of eigenvalues in the cont

So this follows exactly the format on Claude Academy. A small tweak compared to what I had before: the test_case is the plain object of the test case, instead of the end merged prompt.

In [44]:
with open("results/prompt_evaluation.json", "w") as f:
    json.dump(results, f, indent=2)

### Grading

This is the process to assign a score for the the prompt, based on it's output. This step was skipped before for the sake of simplicity, but that's what I'll tackle now.

There are three main ways to grade LLM output.

- **Code:** we can programatically verify the output to ensure things like valid syntax, length, includes or excludes specific words, etc;
- **Human:** we can a human review the outputs and assigning a score for it, good for relevance, conciseness, depth and general response quality;
- **Model:** we can also ask a model to assign a score for the output, or compare versions. This is extremelly flexible, but we can use it for response quality, completeness, safety, quality of following instruction, etc.

### Model Based Grading

We basically do a new API call for the model to evaluate the output against the input.

In [ ]:
def grade_with_model(test_case, output):
    eval_prompt = f"""
You are a math teacher with years of experience in the linear algebra field.
You are able to spot a good explanation about any topics in linear algebra.
Evaluate this answers:

Question: {test_case["question"]}

Output: {output}

Provide your evaluation as a structured JSON object with:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement  
- "reasoning": A concise explanation of your assessment
- "score": A number between 1-10
"""
    response = completion(model=MODEL, messages=[Message(role="user", content=eval_prompt)])
    return json.loads(response.choices[0].message.content)

In [47]:
def run_prompt(test_case):
    prompt = f"""Answer this math related question:\n{test_case['question']}"""

    response = completion(model=MODEL, messages=[Message(role="user", content=prompt)])
    return response.choices[0].message.content

def eval_test_case(test_case):
    output = run_prompt(test_case)

    score = grade_with_model(test_case, output)

    result = {
        "test_case": test_case,
        "output": output,
        "score": score["score"],
    }

    return result

def eval_dataset(dataset):
    return [eval_test_case(test_case) for test_case in dataset]

In [50]:
with open("datasets/prompt_evaluation.json", "r") as f:
    dataset = json.load(f)

results = eval_dataset(dataset)

with open("results/prompt_evaluation.json", "w") as f:
    json.dump(results, f, indent=2)

results

{
  "strengths": [
    "The answer provides a precise and comprehensive definition of a linear transformation, including the key properties (additivity and homogeneity) and their implications.",
    "It correctly emphasizes the distinction between linear and affine transformations, which is a critical conceptual point.",
    "The mention of matrix representation in finite-dimensional spaces connects the abstract definition to practical applications, enhancing understanding."
  ],
  "weaknesses": [
    "The explanation could include a concrete example (e.g., a simple linear transformation like scaling or rotation) to make the concept more tangible.",
    "It might benefit from briefly mentioning the importance of linearity in preserving structure, such as subspaces or solutions to linear equations.",
    "The note about the zero vector mapping is correct but could be derived from the properties rather than stated as a separate point."
  ],
  "reasoning": "The answer is accurate, structu

[{'test_case': {'question': 'What is the definition of a linear transformation in linear algebra?'},
  'output': 'A **linear transformation** (also called a **linear map** or **linear operator**) is a function $ T: V \\rightarrow W $ between two vector spaces $ V $ and $ W $ (over the same field, typically the real or complex numbers) that preserves the operations of **vector addition** and **scalar multiplication**. Specifically, it satisfies the following two properties for all vectors $ \\mathbf{u}, \\mathbf{v} \\in V $ and all scalars $ c $:\n\n1. **Additivity**:  \n   $$\n   T(\\mathbf{u} + \\mathbf{v}) = T(\\mathbf{u}) + T(\\mathbf{v})\n   $$\n2. **Homogeneity**:  \n   $$\n   T(c\\mathbf{u}) = cT(\\mathbf{u})\n   $$\n\nThese properties ensure that $ T $ preserves the linear structure of the vector spaces. If $ V $ and $ W $ are finite-dimensional, linear transformations can be represented by **matrices** with respect to chosen bases. \n\n**Key Note**: Linear transformations map t

In [51]:
with open("results/prompt_evaluation.json", "w") as f:
    json.dump(results, f, indent=2)

There's also a section for Code Grader which I'll skip. It gives a single example of syntax evaluation. The idea is try to parse the output with the expected format. If it errors, give it a 0 score, else a 10. Then just take the average between the model grader and code grader.

## Conclusion

I got 5/6 in the quiz because I missed something about using a faster model for generating the eval dataset, which makes perfect sense come to think about it.

I'll add a short test with `llama3.2:3b`, as suggested by Claude, just to cover using a different faster model.

## Appendix A

Here's the test with faster model as previously mentioned.

In [53]:
import json

def gen_dataset() -> dict:
    prompt = """
Generate a test dataset for prompt evaluation. The dataset will evaluate prompts that assist with math related questions.

Example output:
[
    {
        "question": "Some questions about math"
    },
    ...more questions
]

The answer should be plain JSON, with no aditional commentary, just the raw JSON string.

* Focus on simple questions about linear algebra
* Focus on questions that don't require long winded explanations
* Focus on questions that don't require calculations
* Foocus on questions about theory

Generate 3 objects.
"""
    message = Message(
        role="user",
        content=prompt
    )
    response = completion(
        model="ollama/llama3.2:3b",
        messages=[message]
    )
    raw_json = response.choices[0].message.content
    return json.loads(raw_json)

dataset = gen_dataset()
dataset

[{'question': 'What is the vector space of all 2x2 matrices?'},
 {'question': 'What is the rank-nullity theorem in linear algebra?'},
 {'question': 'What is the difference between a basis and a spanning set in linear algebra?'}]

Yeah it's pretty crazy 0.8s against the previous 11s.

## Appendix B

Although not completely invalidating the eval, running the same inference model as the evaluation model is generally not a good ideal, given the using the same model might result in bias on the evaluation. I didn't fully cover that before, so just like Appendix A, I'll add this here.

In [11]:
import json
from litellm import completion, Message
from constants import MODEL

def grade_with_model(test_case, output):
    eval_prompt = f"""
You are a math teacher with years of experience in the linear algebra field.
You are able to spot a good explanation about any topics in linear algebra.
Evaluate this answers:

Question: {test_case["question"]}

Output: {output}

Provide your evaluation as a structured JSON object with:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement  
- "reasoning": A concise explanation of your assessment
- "score": A number between 1-10

Example Output:
{{
    "strengths": [...]
    "weaknesses": [...]
    "reasoning": <reasoning>
    "score": <score>
}}
"""
    response = completion(
        model="ollama/phi4:14b", # here I use a different model
        messages=[Message(role="user", content=eval_prompt), Message(role="assistant", content="```json\n")],
        stop=["```"],
        format={
            "type": "object",
            "properties": {
                "strengths": {"type": "array", "items": {"type": "string"}},
                "weaknesses": {"type": "array", "items": {"type": "string"}},
                "reasoning": {"type": "string"},
                "score": {"type": "integer"},
            },
            "required": ["strengths", "weaknesses", "reasoning", "score"],
        },
        temperature=0
    )
    print("Response:", response.choices[0].message.content)
    return json.loads(response.choices[0].message.content)

def run_prompt(test_case):
    prompt = f"""Answer this math related question:\n{test_case['question']}"""

    response = completion(model=MODEL, messages=[Message(role="user", content=prompt)])
    return response.choices[0].message.content

def eval_test_case(test_case):
    output = run_prompt(test_case)

    score = grade_with_model(test_case, output)

    result = {
        "test_case": test_case,
        "output": output,
        "score": score["score"],
    }

    return result

def eval_dataset(dataset):
    return [eval_test_case(test_case) for test_case in dataset]

with open("datasets/prompt_evaluation.json", "r") as f:
    dataset = json.load(f)

results = eval_dataset(dataset)

results

Response: {
    "strengths": [
        "The explanation clearly defines a linear transformation with precise mathematical notation, making it accessible to those familiar with linear algebra.",
        "It effectively covers both key properties of linear transformations: additivity and homogeneity, which are essential for understanding the concept.",
        "The explanation connects the definition to broader concepts in linear algebra, such as matrix representation and transformations like scaling and rotation, providing context and relevance."
    ],
    "weaknesses": [
        "The explanation could benefit from a brief example to illustrate the properties of linear transformations, aiding comprehension.",
        "While it mentions that linear transformations preserve vector space structure, a more explicit statement about why this preservation is important could enhance understanding.",
        "The explanation assumes familiarity with terms like 'vector spaces' and 'zero vectors'

[{'test_case': {'question': 'What is the definition of a linear transformation in linear algebra?'},
  'output': 'A **linear transformation** (also called a **linear map**) is a function $ T: V \\rightarrow W $ between two vector spaces $ V $ and $ W $ over the same field (e.g., real numbers $ \\mathbb{R} $ or complex numbers $ \\mathbb{C} $) that satisfies the following two properties for all vectors $ \\mathbf{u}, \\mathbf{v} \\in V $ and all scalars $ c $:\n\n1. **Additivity**:  \n   $$\n   T(\\mathbf{u} + \\mathbf{v}) = T(\\mathbf{u}) + T(\\mathbf{v})\n   $$\n\n2. **Homogeneity**:  \n   $$\n   T(c\\mathbf{u}) = cT(\\mathbf{u})\n   $$\n\nThese properties ensure that $ T $ preserves the structure of vector spaces, including vector addition and scalar multiplication. Linear transformations are fundamental in linear algebra, as they generalize concepts like scaling, rotation, reflection, and projection, and can be represented by matrices when $ V $ and $ W $ are finite-dimensional. A k

Besides the different scores we got, we can see it took longer than before. I think that part of that is an overhead between unloading and loading a model.

1. Spins up Qwen
2. Runs test case 1
3. Spins up Phi
4. Evals test case 1
5. Spins up Qwen
6. Runs test case 2
7. Spins up Phi
8. Evals test case 2
9. Spins up Qwen
10. Runs test case 3
11. Spins up Phi
12. Evals test case 3

Instead of

1. Spins up Qwen
2. Runs test case 1
3. Runs test case 2
4. Runs test case 3
5. Spins up Phi
6. Evals test case 1
7. Evals test case 2
8. Evals test case 3

I'm not sure, but I think this might cause overhead. I'll test changing the flow next.

In [13]:
import json
from litellm import completion, Message
from constants import MODEL

def grade_with_model(result):
    test_case = result["test_case"]
    output = result["output"]

    eval_prompt = f"""
You are a math teacher with years of experience in the linear algebra field.
You are able to spot a good explanation about any topics in linear algebra.
Evaluate this answers:

Question: {test_case["question"]}

Output: {output}

Provide your evaluation as a structured JSON object with:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement  
- "reasoning": A concise explanation of your assessment
- "score": A number between 1-10

Example Output:
{{
    "strengths": [...]
    "weaknesses": [...]
    "reasoning": <reasoning>
    "score": <score>
}}
"""
    response = completion(
        model="ollama/phi4:14b", # here I use a different model
        messages=[Message(role="user", content=eval_prompt), Message(role="assistant", content="```json\n")],
        stop=["```"],
        format={
            "type": "object",
            "properties": {
                "strengths": {"type": "array", "items": {"type": "string"}},
                "weaknesses": {"type": "array", "items": {"type": "string"}},
                "reasoning": {"type": "string"},
                "score": {"type": "integer"},
            },
            "required": ["strengths", "weaknesses", "reasoning", "score"],
        },
        temperature=0
    )
    return json.loads(response.choices[0].message.content)

def run_prompt(test_case):
    prompt = f"""Answer this math related question:\n{test_case['question']}"""

    response = completion(model=MODEL, messages=[Message(role="user", content=prompt)])
    return response.choices[0].message.content

def eval_test_case(test_case):
    output = run_prompt(test_case)

    result = {
        "test_case": test_case,
        "output": output,
    }

    return result

def eval_dataset(dataset):
    return [eval_test_case(test_case) for test_case in dataset]

def grade_results(results):
    for result in results:
        grade = grade_with_model(result)
        result["score"] = grade["score"]

with open("datasets/prompt_evaluation.json", "r") as f:
    dataset = json.load(f)

results = eval_dataset(dataset)
grade_results(results)
results

[{'test_case': {'question': 'What is the definition of a linear transformation in linear algebra?'},
  'output': 'A **linear transformation** in linear algebra is a function $ T: V \\rightarrow W $ between two vector spaces $ V $ and $ W $ over the same field $ \\mathbb{F} $ that satisfies the following two properties for all vectors $ \\mathbf{u}, \\mathbf{v} \\in V $ and all scalars $ a \\in \\mathbb{F} $:\n\n1. **Additivity**:  \n   $$\n   T(\\mathbf{u} + \\mathbf{v}) = T(\\mathbf{u}) + T(\\mathbf{v})\n   $$\n\n2. **Homogeneity**:  \n   $$\n   T(a\\mathbf{u}) = aT(\\mathbf{u})\n   $$\n\nThese properties can be combined into a single condition:  \n$$\nT(a\\mathbf{u} + b\\mathbf{v}) = aT(\\mathbf{u}) + bT(\\mathbf{v}),\n$$\nfor all scalars $ a, b \\in \\mathbb{F} $ and vectors $ \\mathbf{u}, \\mathbf{v} \\in V $. This ensures that $ T $ preserves **linear combinations** of vectors, making it a structure-preserving map between vector spaces. Linear transformations are also called **lin

Small difference, 2m42s to 2m25s. Also, apparently the plain Phi4 model I'm using doesn't do reasoning. Since the prompt ask for reasoning it probably goes through a small thinking process.